# 6.2: Parameter Management

In [1]:
import torch
from torch import nn

In [2]:
net = nn.Sequential(nn.LazyLinear(8),
                    nn.ReLU(),
                    nn.LazyLinear(1))

X = torch.rand(size=(2,4))
net(X).shape

c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


torch.Size([2, 1])

## 6.2.1: Parameter Access

In [3]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.3427,  0.2117,  0.2937, -0.0459, -0.2353, -0.1360, -0.3472, -0.1405]])),
             ('bias', tensor([0.1777]))])

### Target Parameter Extraction

In [4]:
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.1777]))

In [5]:
net[2].weight.grad == None

True

Note that weight gradient is None because we have not invoked backpropagation for this network yet

### All Parameters at Once

In [6]:
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

## 6.2.2: Tied Parameters

In [7]:
# We need to give the shared layer a name so that we can refer to its parameters
shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))
net(X)
# Check whether parameters are the same
print(net[2].weight.data[0] == net[4].weight.data[0]) # Confirm axis 0 (only axis) is the same
net[2].weight.data[0,0] = 100
# Make sure that they are actually the same object rather than just having the same value
print(net[2].weight.data[0] == net[4].weight.data[0]) 

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


This shows the parameters of the second and third later are tied (represented by exact same tensor). Thus, if we change one of the parameters, the other one changes too.

# Summary:

We have sevreal ways of accessing and tying parameters.